In [326]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Data Preperation

###Data profiling + Understanding data

Meta Data: 5 excercies are tracked including bench press, Deadlift, Overheadpress, Barbell row, Squat. A watch with a gyroscope and an accelormeter tracks the moments associated. There are 5 participants, data is divided amongst each participant by exercise (each csv is semi structured)

In [327]:
import pandas as pd
file1= pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-bench-heavy2-rpe8_MetaWear_2019-01-11T16.10.08.270_C42732BE255C_Accelerometer_12.500Hz_1.4.4.csv")
file2= pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-bench-heavy2-rpe8_MetaWear_2019-01-11T16.10.08.270_C42732BE255C_Gyroscope_25.000Hz_1.4.4.csv")
filen= pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-rest-standing_MetaWear_2019-01-18T18.25.39.382_C42732BE255C_Accelerometer_12.500Hz_1.4.41.csv")

In [328]:
filen.head()

,epoch (ms),time (01:00),elapsed (s),x-axis (g),y-axis (g),z-axis (g)
0,1547918739909,2019-01-19 18:25:39.908,0.00,0.951,0.110,0.193
1,1547918739989,2019-01-19 18:25:39.988,0.08,0.971,0.105,0.196
2,1547918740069,2019-01-19 18:25:40.068,0.16,0.954,0.081,0.196
3,1547918740149,2019-01-19 18:25:40.148,0.24,0.988,0.067,0.233
4,1547918740229,2019-01-19 18:25:40.228,0.32,0.971,0.085,0.242


In [329]:
filen.columns

Index(['epoch (ms)', 'time (01:00)', 'elapsed (s)', 'x-axis (g)', 'y-axis (g)',
       'z-axis (g)'],
      dtype='object')

All scattered csv files have 6 columns: 'epoch (ms)', 'time (01:00)', 'elapsed (s)', 'x-axis (g)', 'y-axis (g)',
       'z-axis (g)'

In [330]:
filen.dtypes

,0
epoch (ms),int64
time (01:00),object
elapsed (s),float64
x-axis (g),float64
y-axis (g),float64
z-axis (g),float64


In [331]:
from glob import glob #allows to find all the pathnames matching a specified pattern
files = glob ("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/*.csv")
files #lists the file paths using glob
len(files) #187 csv files found

187

In [332]:
files[0]

'/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-dead-heavy_MetaWear_2019-01-15T20.35.27.174_C42732BE255C_Gyroscope_25.000Hz_1.4.4.csv'

Since csv are differentiated based on excercise type, participant, category whether its a heavy set or not, i need to extract that data from the file names itself first before merging the individual csv

####Extracting data from file paths itself

In [333]:
data_path = "/content/drive/MyDrive/Fitness Tracker/Raw Data Files/"
f = files[0]
f

'/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-dead-heavy_MetaWear_2019-01-15T20.35.27.174_C42732BE255C_Gyroscope_25.000Hz_1.4.4.csv'

In [334]:
participant = f.split("-")[0].replace("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/", "")
label = f.split("-")[1]
category = f.split("-")[2].split("_")[0]

In [335]:
df_1 = pd.read_csv(files[0]) #reading file 1 again

#feature eng
df_1["participant"] = participant
df_1["label"] = label
df_1["category"] = category

df_1 #(voila)

,epoch (ms),time (01:00),elapsed (s),x-axis (deg/s),y-axis (deg/s),z-axis (deg/s),participant,label,category
0,1547580927366,2019-01-15T20:35:27.366,0.00,8.049,-0.122,20.610,A,dead,heavy
1,1547580927406,2019-01-15T20:35:27.406,0.04,6.037,2.561,18.354,A,dead,heavy
2,1547580927446,2019-01-15T20:35:27.446,0.08,-8.841,23.049,0.854,A,dead,heavy
3,1547580927486,2019-01-15T20:35:27.486,0.12,-17.256,21.585,-1.890,A,dead,heavy
4,1547580927526,2019-01-15T20:35:27.526,0.16,-2.439,-17.561,-6.524,A,dead,heavy
...,...,...,...,...,...,...,...,...,...
405,1547580943566,2019-01-15T20:35:43.566,16.20,2.195,16.037,16.646,A,dead,heavy
406,1547580943606,2019-01-15T20:35:43.606,16.24,2.195,16.098,15.305,A,dead,heavy
407,1547580943646,2019-01-15T20:35:43.646,16.28,5.671,-1.463,10.183,A,dead,heavy
408,1547580943686,2019-01-15T20:35:43.686,16.32,5.854,-2.988,9.512,A,dead,heavy


####Replicating and looping the code tot he rest of the files

In [336]:
acceleratometer_df = pd.DataFrame()
gyroscope_df = pd.DataFrame()
acc_df = 1
gyro_df = 1

for f in files:
  #creating individual df from csv
  participant = f.split("-")[0].replace("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/", "")
  label = f.split("-")[1]
  category = f.split("-")[2].split("_")[0]

  df = pd.read_csv(f)
  #feature eng
  df["participant"] = participant
  df["label"] = label
  df["category"] = category

  #differenciating Accelerometer and Gyroscope data
  if "Accelerometer" in f:
    df["set"]= acc_df
    acc_df += 1
    acceleratometer_df = pd.concat([acceleratometer_df, df]) #pd.concat()
  elif "Gyroscope" in f:
    df["set"]= gyro_df
    gyro_df += 1
    gyroscope_df = pd.concat([gyroscope_df, df])

In [337]:
def set_index_to_epoch (df):
  df["epoch (ms)"] = pd.to_datetime(df["epoch (ms)"], unit= "ms")
  df.dtypes
  df.index = df["epoch (ms)"]
  return df

acceleratometer_df = set_index_to_epoch(acceleratometer_df)
gyroscope_df = set_index_to_epoch(gyroscope_df)
gyroscope_df.head()

,epoch (ms),time (01:00),elapsed (s),x-axis (deg/s),y-axis (deg/s),z-axis (deg/s),participant,label,category,set
epoch (ms),,,,,,,,,,
2019-01-15 19:35:27.366,2019-01-15 19:35:27.366,2019-01-15T20:35:27.366,0.00,8.049,-0.122,20.610,A,dead,heavy,1
2019-01-15 19:35:27.406,2019-01-15 19:35:27.406,2019-01-15T20:35:27.406,0.04,6.037,2.561,18.354,A,dead,heavy,1
2019-01-15 19:35:27.446,2019-01-15 19:35:27.446,2019-01-15T20:35:27.446,0.08,-8.841,23.049,0.854,A,dead,heavy,1
2019-01-15 19:35:27.486,2019-01-15 19:35:27.486,2019-01-15T20:35:27.486,0.12,-17.256,21.585,-1.890,A,dead,heavy,1
2019-01-15 19:35:27.526,2019-01-15 19:35:27.526,2019-01-15T20:35:27.526,0.16,-2.439,-17.561,-6.524,A,dead,heavy,1


In [338]:
gyroscope_df.columns

Index(['epoch (ms)', 'time (01:00)', 'elapsed (s)', 'x-axis (deg/s)',
       'y-axis (deg/s)', 'z-axis (deg/s)', 'participant', 'label', 'category',
       'set'],
      dtype='object')

In [339]:
def drop_dt_columns_in_df(df):
  df = df.drop(['epoch (ms)', 'time (01:00)', 'elapsed (s)'], axis=1)
  return df
gyroscope_df = drop_dt_columns_in_df(gyroscope_df)
acceleratometer_df = drop_dt_columns_in_df(acceleratometer_df)

In [340]:
gyroscope_df.columns

Index(['x-axis (deg/s)', 'y-axis (deg/s)', 'z-axis (deg/s)', 'participant',
       'label', 'category', 'set'],
      dtype='object')

In [341]:
df_merged = pd.concat([acceleratometer_df.loc[:,['x-axis (g)', 'y-axis (g)', 'z-axis (g)']], gyroscope_df], axis=1) #axis 1 is col wise / horizontal
df = df_merged.copy()
df = df.drop(axis=1, columns=["set"])
df.head()
df.shape

(69677, 9)

In [342]:
df.dropna().shape

(1119, 9)

In [343]:
def NA_percent (df):
  return (df.isna().sum()/df.shape[0])*100
NA_percent(df)

,0
x-axis (g),66.161000
y-axis (g),66.161000
z-axis (g),66.161000
x-axis (deg/s),32.233018
y-axis (deg/s),32.233018
z-axis (deg/s),32.233018
participant,32.233018
label,32.233018
category,32.233018


###Resampling

gyroscope typically samples faster and has a higher maximum data rate than the accelerometer. Hence more data in gyroscope than accelerometer. Because your gyroscope and accelerometer sample data at different speeds, resampling is the perfect way to force them to use the exact same time intervals so they match up perfectly.

In [344]:
#experimanting different resampling parameters

In [345]:
df1 = df.iloc[:1000].select_dtypes(include=['number']).resample(rule="S").mean()
df1
#rule="S" resample by each second and squash everything in to row that contains data which has mean data for that second

/tmp/ipykernel_1140/418544541.py:1: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df1 = df.iloc[:1000].select_dtypes(include=['number']).resample(rule="S").mean()


,x-axis (g),y-axis (g),z-axis (g),x-axis (deg/s),y-axis (deg/s),z-axis (deg/s)
epoch (ms),,,,,,
2019-01-11 15:08:04,NaN,NaN,NaN,-9.695500,-1.798500,4.573500
2019-01-11 15:08:05,-0.002222,0.969333,-0.071222,1.007320,-0.731720,-0.399960
2019-01-11 15:08:06,-0.099231,0.899462,-0.163846,6.978040,2.417120,-11.256200
2019-01-11 15:08:07,-0.204750,1.055917,-0.156333,1.317040,-2.083000,2.248840
2019-01-11 15:08:08,-0.079462,0.920462,-0.085692,-9.826840,-4.697560,8.356160
...,...,...,...,...,...,...
2019-01-11 15:10:18,-0.159308,1.022154,-0.169462,-4.687760,-0.229240,-2.634160
2019-01-11 15:10:19,-0.136167,0.938417,-0.129417,-4.741400,1.731800,14.529320
2019-01-11 15:10:20,-0.003769,0.938615,-0.106846,6.426880,-5.836560,-4.370720


In [346]:
df2 = df.iloc[:1000].select_dtypes(include=['number']).resample(rule="200ms").mean()
df2.shape

(689, 6)

In [347]:
df.columns

Index(['x-axis (g)', 'y-axis (g)', 'z-axis (g)', 'x-axis (deg/s)',
       'y-axis (deg/s)', 'z-axis (deg/s)', 'participant', 'label', 'category'],
      dtype='object')

In [348]:
df.dropna().shape

(1119, 9)

In [349]:
#resampling

sampling = {'x-axis (g)':"mean", 'y-axis (g)':"mean", 'z-axis (g)':"mean", 'x-axis (deg/s)':"mean",
       'y-axis (deg/s)':"mean", 'z-axis (deg/s)':"mean", 'participant':"last", 'label':"last", 'category':"last"}
df = df.resample(rule="100ms").apply(sampling) #100ms is selected as its the optimal window to consider before start losing critical movement details.
df.shape

(7864289, 9)

In [355]:
df_final = df.dropna()
df_final.shape #17912 rows

(17912, 9)

#Train-test split

We are going to use 3 particpants A B C for training, D for Validating, E for testing

In [359]:
df = df_final.copy()
df.head()

,x-axis (g),y-axis (g),z-axis (g),x-axis (deg/s),y-axis (deg/s),z-axis (deg/s),participant,label,category
epoch (ms),,,,,,,,,
2019-01-11 15:08:05.300,0.0135,0.9770,-0.071,-1.524333,3.069333,4.573000,B,bench,heavy1
2019-01-11 15:08:05.400,-0.0110,0.9700,-0.086,0.732000,-2.104000,-0.518500,B,bench,heavy1
2019-01-11 15:08:05.500,0.0080,0.9710,-0.073,-3.292333,-0.081333,3.963667,B,bench,heavy1
2019-01-11 15:08:05.600,0.0120,0.9960,-0.051,-8.597500,-2.774500,-1.006000,B,bench,heavy1
2019-01-11 15:08:05.700,-0.0040,0.9595,-0.071,9.999667,1.423000,-1.687000,B,bench,heavy1


In [371]:
train = df.loc[df["participant"].isin(["A","E","C"])]
train.shape

(14145, 9)

In [372]:
val = df.loc[df["participant"].isin(["D"])]
val.shape

(2092, 9)

In [373]:
test = df.loc[df["participant"].isin(["B"])]
test.shape

(1675, 9)

Train: ~79% (14,145 rows) — Plenty of data for your model to learn the intricacies of the barbell movements.
Validation (Val): ~12% (2,092 rows)
Test: ~9% (1,675 rows)

In [378]:
#Exporting intermediate files

train.to_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files/train.csv")
val.to_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files/val.csv")
test.to_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files/test.csv")